[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VirtualFlyBrain/neurofly2026-workshop/blob/main/python/01_Discovery.ipynb)

# 01 · Discovery — finding neurons across datasets
**Problem P1:** *Find every instance of a neuron type across all datasets.*

VFB classifies neurons from every source with one ontology ([Drosophila Anatomy Ontology](https://www.ebi.ac.uk/ols4/ontologies/fbbt)), so a single query for a *type* returns individuals from FlyWire, hemibrain, BANC, male-CNS, MANC, optic-lobe and CATMAID at once — no wrangling nomenclature.

In [ ]:
# Run once per Colab session (skip if already installed locally).
# vfb_connect pulls two legacy deps (jsonpath-rw, colormath) that only build on an
# older setuptools, so pin it first — otherwise the install fails on Colab:
%pip install -q "setuptools<58" wheel
%pip install -q vfb_connect navis navis-flybrains neuprint-python python-catmaid

In [ ]:
# Modern entry point: a ready-made singleton that wraps VFB's public servers.
from vfb_connect import vfb
import pandas as pd

# (Legacy equivalent, as used in the 2024 notebooks:
#   from vfb_connect.cross_server_tools import VfbConnect; vc = VfbConnect() )

## Route A — `vfb_connect`
Find the type name/symbol first via the [VFB search](https://virtualflybrain.org), then pull its instances. Substitute a type *you* care about.

**Worked example:** the DA1 lateral projection neuron (`DA1 lPN`, `FBbt_00067363`) — a well-studied olfactory PN. In VFB it resolves to **68 individuals** spanning hemibrain, FlyWire, BANC, male-CNS and FAFB. (It's *absent* from MANC and the optic lobe — it's a central-brain neuron, not in the nerve cord or optic lobe — a good reminder that not every cell is in every dataset.)

In [ ]:
# All individual neurons of a given type, across every dataset.
neuron_type = 'adult antennal lobe projection neuron DA1 lPN'   # DA1 lPN — change me
df = vfb.get_instances(neuron_type)   # returns a DataFrame
df.head(10)

In [ ]:
# Which datasets is this type represented in? (column name may be 'data_source' or 'dataset')
col = 'data_source' if 'data_source' in df.columns else 'dataset'
df[col].value_counts()

### Find neurons by location
The ontology also lets you ask 'what's *in* a region' — parts and overlapping cells.

In [ ]:
region_terms = vfb.get_terms_by_region('fan-shaped body')  # try 'medulla', 'mushroom body'
pd.DataFrame(region_terms).head()

### What's *new* since 2024?
The 2026-era connectomes already show up in the instance table above. Filter to the neurons that come from datasets added since the last workshop (BANC whole-CNS, male-CNS):

In [ ]:
new_sets = df[df['data_source'].str.contains('BANC|MaleCNS|male', case=False, na=False)]
print(f'{len(new_sets)} DA1 lPN individuals in datasets new since 2024')
new_sets[['label', 'id', 'data_source']].head(10)

#### Route B — VFB MCP tool (in your LLM)

Set up once via [`../no-code/MCP_setup.md`](../no-code/MCP_setup.md), then ask:

> "Search VFB for the neuron type DA1 lPN and list every individual across all datasets with their dataset and VFB ID."

#### Route C — chat.virtualflybrain.org

No setup — open [chat.virtualflybrain.org](https://chat.virtualflybrain.org) and ask:

> "Where do I find DA1 lPN neurons in VFB, and which connectomes have them?"

> **When to reach for which** — chat/MCP to nail the right type name quickly; the API when you need the full instance table to feed visualisation (03) or connectivity (04).

### 🧪 Your turn
Pick a neuron type from your own work. How many datasets is it in? Does **BANC** or **male-CNS** add instances that weren't available in 2024?